In [54]:
from pathlib import Path
import pandas as pd

In [55]:
PROCESSED_PATH = Path("../data/processed/clean_parquet")

dfs = {}

for file in PROCESSED_PATH.glob("*.parquet"):
    dfs[file.stem] = pd.read_parquet(file)

In [56]:
circuit_type_map = {
    # Street circuits
    "Monaco": "Street",
    "Singapore": "Street",
    "Baku": "Street",
    "Jeddah": "Street",

    # Permanent
    "Silverstone": "Permanent",
    "Spa": "Permanent",
    "Monza": "Permanent",

    # Hybrid / semi-permanent
    "Melbourne": "Hybrid",
    "Las Vegas": "Street"
}

In [57]:
dfs["circuits"]["CircuitType"] = dfs["circuits"]["name"].map(circuit_type_map).fillna("Permanent")

In [58]:
status_map = {
    "Finished": "Finished",

    # Mechanical
    "Engine": "Mechanical",
    "Gearbox": "Mechanical",
    "Hydraulics": "Mechanical",
    "Electrical": "Mechanical",

    # Accident
    "Collision": "Accident",
    "Accident": "Accident",
    "Spun off": "Accident",

    # Driver error
    "Puncture": "Driver Error",
    "Retired": "Driver Error",
    "Withdrew": "Driver Error",

    # Everything else
}

In [59]:
dfs["status"]["StatusCategory"] = dfs["status"]["status"].map(status_map).fillna("Other")

In [60]:
driver_season = (
    dfs["results"]
    .merge(dfs["races"][["raceId", "year"]], on="raceId")
    .groupby(["driverId", "year", "constructorId"])
    .agg(points=("points", "sum"))
    .reset_index()
)

In [61]:
driver_season["teammate_points"] = driver_season.groupby(
    ["constructorId", "year"]
)["points"].transform("sum") - driver_season["points"]

In [62]:
driver_season["TeammatePointsDelta"] = (
    driver_season["points"] - driver_season["teammate_points"]
)

In [63]:
driver_season["TeammateAvgPoints"] = (
    driver_season.groupby(["constructorId", "year"])["points"].transform("mean")
)

driver_season["TeammatePointsDelta"] = (
    driver_season["points"] - driver_season["TeammateAvgPoints"]
)

In [64]:
qual = dfs["qualifying"].merge(
    dfs["races"][["raceId", "year"]],
    on="raceId"
)

qual_stats = (
    qual.groupby(["driverId", "year"])
    .agg(avg_grid=("position", "mean"))
    .reset_index()
)

In [65]:
pit = dfs["pit_stops"].sort_values(["raceId", "driverId", "stop"])

In [66]:
pit["next_lap"] = pit.groupby(["raceId", "driverId"])["lap"].shift(-1)

In [67]:
stints = pit.copy()

stints["stint_length"] = stints["next_lap"] - stints["lap"]

In [68]:
max_lap = dfs["lap_times"].groupby(
    ["raceId", "driverId"]
)["lap"].max().reset_index()

In [69]:
race_results = dfs["results"].merge(
    dfs["races"][["raceId", "year", "date"]],
    on="raceId"
)

In [70]:
race_results = race_results.sort_values(["driverId", "date"])

In [71]:
race_results["rolling_points_5"] = (
    race_results.groupby("driverId")["points"]
    .transform(lambda x: x.shift().rolling(5, min_periods=1).mean())
)

In [72]:
race_results[["grid", "positionOrder"]].dtypes

grid             object
positionOrder     int64
dtype: object

In [73]:
race_results["grid"] = pd.to_numeric(race_results["grid"], errors="coerce")
race_results["positionOrder"] = pd.to_numeric(race_results["positionOrder"], errors="coerce")

In [74]:
race_results["positions_gained"] = (
    race_results["grid"] - race_results["positionOrder"]
)

In [75]:
race_results["positions_gained"] = (
    race_results["grid"].where(race_results["grid"].notna())
    - race_results["positionOrder"]
)

In [29]:
race_results[["grid", "positionOrder", "positions_gained"]].head()

,grid,positionOrder,positions_gained
370,4.0,3,1.0
391,4.0,2,2.0
413,2.0,2,0.0
435,4.0,2,2.0
457,2.0,2,0.0


In [76]:
race_results

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,fastestLapTime,fastestLapSpeed,statusId,gap_ms,gap_laps,gap_type,year,date,rolling_points_5,positions_gained
370,371,36,1,1,2,4.0,3,3,3,6.0,...,1:26.351,221.083,1,None,None,None,2007,2007-03-18,NaN,1.0
391,392,37,1,1,2,4.0,2,2,2,8.0,...,1:36.701,206.355,1,None,None,None,2007,2007-04-08,6.000000,2.0
413,414,38,1,1,2,2.0,2,2,2,8.0,...,1:34.270,206.674,1,None,None,None,2007,2007-04-15,7.000000,0.0
435,436,39,1,1,2,4.0,2,2,2,8.0,...,1:22.876,202.205,1,None,None,None,2007,2007-05-13,7.333333,2.0
457,458,40,1,1,2,2.0,2,2,2,8.0,...,1:15.372,159.528,1,None,None,None,2007,2007-05-27,7.500000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27317,27323,1172,866,215,41,16.0,14,14,14,0.0,...,1:34.058,None,11,None,None,None,2026,2026-05-03,1.333333,2.0
27347,27353,1173,866,215,41,9.0,22,W,22,0.0,...,None,None,6,None,None,None,2026,2026-05-24,1.000000,-13.0
27354,27360,1174,866,215,41,15.0,7,7,7,6.0,...,1:15.908,None,1,None,None,None,2026,2026-06-07,0.800000,8.0
27379,27385,1175,866,215,41,11.0,10,10,10,1.0,...,1:21.914,None,11,None,None,None,2026,2026-06-14,1.200000,1.0


In [30]:
race_results["rolling_positions_gained_5"] = (
    race_results.groupby("driverId")["positions_gained"]
    .transform(lambda x: x.shift().rolling(5, min_periods=1).mean())
)

In [31]:
OUTPUT = Path("../data/feature_layer")
OUTPUT.mkdir(exist_ok=True)

race_results.to_parquet(OUTPUT / "race_features.parquet", index=False)
driver_season.to_parquet(OUTPUT / "driver_season_features.parquet", index=False)
stints.to_parquet(OUTPUT / "stint_features.parquet", index=False)